In [1]:
import glob
import h5py
import importlib
import IPython.display as ipd
import soxr
import numpy as np
import os
import pandas as pd
import pickle
import soundfile as sf
import src.spatial_attn_lightning as binaural_lightning
import src.audio_transforms as at
importlib.reload(binaural_lightning)
import sys
import torch
import tqdm
import yaml

from pathlib import Path
from pytorch_lightning import Trainer
from scipy import signal
from scipy.io.wavfile import read, write

sys.path.append('../')
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [ ]:
h5_file = '/om2/user/rphess/Auditory-Attention/Room_HRIRs_50kHz.h5'
hrirs = h5py.File(h5_file, 'r', swmr=True)

In [3]:
speaker_dir = '/om2/user/imgriff/datasets/commonvoice_9_en/3000ms/stimSR_50000/cv_9_en/subsets/'
speaker_paths = glob.glob(speaker_dir + 'train_core/*')
speaker_h5 = h5py.File(speaker_paths[0], 'r', swmr=True)

In [4]:
def spatialize_sound(y, brir):
    """
    This takes a left-aligned BRIR and convolves it with a left-padded signal
    (using "valid" padding) to produce the same output as "same" padding with
    a center-aligned BRIR. It is faster to pad the signal than it is to pad
    the BRIR.
    
    Args
    ----
    y (np.ndarray): monoaural waveform with shape [timesteps]
    brir (np.ndarray): binaural room impulse response with shape [timesteps, 2]
    
    Returns
    -------
    y_spatialized (np.ndarray): binaural waveform with shape [timesteps, 2]
    """
    y_padded = np.pad(y, (brir.shape[0] - 1, 0))
    y_spatialized_l = signal.convolve(y_padded, brir[:, 0], mode='valid', method='direct')
    y_spatialized_r = signal.convolve(y_padded, brir[:, 1], mode='valid', method='direct')
    return np.stack([y_spatialized_l, y_spatialized_r]).T

Want to use code from above to generate validation/test sets to evaluate the model's performance with two speakers at the same location and 90 degrees apart

In [5]:
rand_room = np.random.choice(list(hrirs.keys()))
room = hrirs[rand_room]
listener_locs = list(room.keys())
listener = room[listener_locs[0]]
impulse_response = listener['irs']

In [6]:
ixs_0_elev = [ix for ix, elev in enumerate(listener['elevation']) if elev == 0]

# find five indexes that correspond to 5 azimuths that are 45 degrees apart out of the ixs_0_elev
ixs = []
for ix in ixs_0_elev:
    if listener['azimuth'][ix] in (0, 45, 90, 315, 270):
        ixs.append(ix)

In [7]:
# picking two different speakers at random
speaker1_int = np.random.randint(0, len(speaker_h5['label_talker_int']))
speaker1_label = speaker_h5['label_talker_int'][speaker1_int]
same = True
while same:
    speaker2_int = np.random.randint(0, len(speaker_h5['label_talker_int']))
    if speaker_h5['label_talker_int'][speaker2_int] != speaker1_label:
        same = False

In [8]:
# spatializing selected speakers
speaker1 = speaker_h5['signal'][speaker1_int]
speaker2 = speaker_h5['signal'][speaker2_int]
spatialized_samples = {'speaker1': dict(), 'speaker2': dict()}
for ix in ixs:
    azim = int(listener['azimuth'][ix])
    elev = int(listener['elevation'][ix])
    brir = impulse_response[ix]
    spatialized1 = spatialize_sound(speaker1, brir)
    spatialized2 = spatialize_sound(speaker2, brir)
    spatialized_samples['speaker1'][(azim, elev)] = spatialized1
    spatialized_samples['speaker2'][(azim, elev)] = spatialized2

Automate all code above to make multiple samples and test speed in a model

In [3]:
# load in and randomly initialize model
config_path = "config/binaural_attn/word_task_both_cue.yml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 10
config['hparas']['batch_size'] = 64
config['audio']['rep_kwargs']['rep_on_gpu'] = True

In [4]:
module = binaural_lightning.BinauralAttentionModule(config)

num_classes={'num_words': 800}
Model performing word task
cochlea_filt {'sr': 50000, 'env_sr': 10000, 'n_channels': 40, 'low_lim': 40, 'use_pad': True, 'binaural': True, 'rep_on_gpu': True, 'center_crop': True, 'out_dur': 2, 'impulse_len': 0.25, 'env_extraction_type': 'Half-wave Rectification', 'downsampling_type': 'TorchTransformsResample', 'downsampling_kwargs': {'lowpass_filter_width': 64, 'rolloff': 0.9475937167399596, 'resampling_method': 'kaiser_window', 'beta': 14.769656459379492}} coch_p3 {'scale': 1, 'offset': 1e-07, 'clip_value': 5, 'power': 0.3}
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [6]:
# randomize values of model parameters
for p in module.parameters():
    p.data = torch.nn.parameter.Parameter(torch.randn_like(p))

In [12]:
def spatialize_sound(y, brir):
    """
    This takes a left-aligned BRIR and convolves it with a left-padded signal
    (using "valid" padding) to produce the same output as "same" padding with
    a center-aligned BRIR. It is faster to pad the signal than it is to pad
    the BRIR.
    
    Args
    ----
    y (np.ndarray): monoaural waveform with shape [timesteps]
    brir (np.ndarray): binaural room impulse response with shape [timesteps, 2]
    
    Returns
    -------
    y_spatialized (np.ndarray): binaural waveform with shape [timesteps, 2]
    """
    y_padded = np.pad(y, (brir.shape[0] - 1, 0))
    y_spatialized_l = signal.convolve(y_padded, brir[:, 0], mode='valid', method='direct')
    y_spatialized_r = signal.convolve(y_padded, brir[:, 1], mode='valid', method='direct')
    return np.stack([y_spatialized_l, y_spatialized_r]).T

In [12]:
h5_file = '/om2/user/rphess/Auditory-Attention/Room_HRIRs_50kHz.h5'
hrirs = h5py.File(h5_file, 'r', swmr=True)

rand_room = np.random.choice(list(hrirs.keys()))
room = hrirs[rand_room]
listener_locs = list(room.keys())
listener = room[listener_locs[0]]
impulse_response = listener['irs']

ixs_0_elev = [ix for ix, elev in enumerate(listener['elevation']) if elev == 0]

# find five indexes that correspond to 5 azimuths that are 45 degrees apart out of the ixs_0_elev
ixs = []
for ix in ixs_0_elev:
    if listener['azimuth'][ix] in (0, 90):
        ixs.append(ix)
ixs

[1481, 1683]

In [13]:
speaker_dir = '/om2/user/imgriff/datasets/commonvoice_9_en/3000ms/stimSR_50000/cv_9_en/subsets/'
speaker_paths = glob.glob(speaker_dir + 'train_core/*')
speaker_h5 = h5py.File(speaker_paths[0], 'r', swmr=True)

In [14]:
speaker_h5.keys()

<KeysViewHDF5 ['client_int', 'dataset_split', 'index_file', 'index_manifest', 'label_talker_int', 'label_talker_sex_int', 'label_word_int', 'parent_timestamp_signal_end', 'parent_timestamp_signal_start', 'parent_timestamp_target_end', 'parent_timestamp_target_start', 'signal', 'sr']>

In [31]:
# now generate 200 examples of spatialized speech with 2 speakers
for _ in tqdm.tqdm(range(200)):
    # picking two different speakers at random
    speaker1_int = np.random.randint(0, len(speaker_h5['label_talker_int']))
    speaker1_label = speaker_h5['label_talker_int'][speaker1_int]
    same = True
    while same:
        speaker2_int = np.random.randint(0, len(speaker_h5['label_talker_int']))
        if speaker_h5['label_talker_int'][speaker2_int] != speaker1_label:
            same = False

    # spatializing selected speakers
    speaker1 = speaker_h5['signal'][speaker1_int]
    speaker2 = speaker_h5['signal'][speaker2_int]
    spatialized_samples = {'speaker1': dict(), 'speaker2': dict()}
    for ix in ixs:
        azim = int(listener['azimuth'][ix])
        elev = int(listener['elevation'][ix])
        brir = impulse_response[ix]
        spatialized1 = spatialize_sound(speaker1, brir)
        spatialized2 = spatialize_sound(speaker2, brir)
        spatialized_samples['speaker1'][(azim, elev)] = spatialized1
        spatialized_samples['speaker2'][(azim, elev)] = spatialized2

    # grab cue and label then pass through model
    label = speaker_h5['label_word_int'][speaker1_int]
    fg = torch.from_numpy(spatialized_samples['speaker1'][(90, 0)].T).view(1, 2, -1)
    bg = torch.from_numpy(spatialized_samples['speaker2'][(0, 0)].T).view(1, 2, -1)
    module(fg, bg, None)

 12%|█▏        | 23/200 [02:40<20:32,  6.97s/it]


KeyboardInterrupt: 

Creating an h5 with each word spatialized at each speaker location

In [2]:
print("Loading speaker array room BRIRs")
list_data_dict = []
for elev in [-20, -10, 0, 10, 20, 30, 40]:
    for azim in np.arange(0, 360, 5):
        data_dict = {
            'azim': azim,
            'elev': elev,
            'brir': [],
        }
        for ear in ['l', 'r']:
            basename = f'{elev}elev_{azim}az_2.47x2.60y2.00z_{ear}.wav'
            if elev >= 0:
                fn = os.path.join('/om/user/francl/Room_Simulator_20181115_Rebuild/room_HRIRs/', basename)
            else:
                fn = os.path.join('/om/user/francl/Room_Simulator_20181115_Rebuild/room_HRIRs/neg_elevs/', basename)
            assert os.path.exists(fn)
            brir, sr_src = sf.read(fn)
            sr = 50000
            brir = soxr.resample(brir.astype(np.float32), sr_src, sr)
            data_dict['brir'].append(brir)
        data_dict['sr'] = sr
        data_dict['brir'] = np.stack(data_dict['brir']).T
        list_data_dict.append(data_dict)
df_brir = pd.DataFrame(list_data_dict)
df_brir_room = df_brir[np.logical_and.reduce([
    df_brir['azim'] % 10 == 0,
    ~(np.logical_and(df_brir['azim'] > 90, df_brir['azim'] < 270)),
    df_brir['elev'] >= 0,
])].reset_index()
print(f"Loaded speaker array room BRIRs ({len(df_brir_room)})")

Loading speaker array room BRIRs
Loaded speaker array room BRIRs (95)


In [3]:
word_dir = '/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/foreground_swc/'
word_paths = glob.glob(word_dir + '*')
word_data = pickle.load(open(word_paths[-1], 'rb'))


In [4]:
word_data

,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,raw_clip_dur_in_s,raw_clip_end_in_s,raw_clip_start_in_s,raw_src_fn,raw_total_file_duration_in_s,split,split_int,sr,src_fn,total_file_duration_in_s,word
0,a-c-norman,3.0,3.0,0.0,swc,0.32,2094.94,2094.62,/scratch2/weka/mcdermott/msaddler/swc/english/...,2175.444172,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,above
1,jebjoya,3.0,3.0,0.0,swc,0.49,1715.87,1715.38,/scratch2/weka/mcdermott/msaddler/swc/english/...,2793.356190,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,according
2,caninedoubletake,3.0,3.0,0.0,swc,0.36,169.03,168.67,/scratch2/weka/mcdermott/msaddler/swc/english/...,987.438730,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,across
3,karltalk,3.0,3.0,0.0,swc,0.60,2429.51,2428.91,/scratch2/weka/mcdermott/msaddler/swc/english/...,4802.892336,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,action
4,s-whistler,3.0,3.0,0.0,swc,0.80,1149.87,1149.07,/scratch2/weka/mcdermott/msaddler/swc/english/...,4463.715556,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,activities
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
371,incledon,3.0,3.0,0.0,swc,0.69,231.72,231.03,/scratch2/weka/mcdermott/msaddler/swc/english/...,273.573152,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,world
372,ama1016,3.0,3.0,0.0,swc,0.42,193.47,193.05,/scratch2/weka/mcdermott/msaddler/swc/english/...,273.502041,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,writing
373,zanimum,3.0,3.0,0.0,swc,0.27,13.92,13.65,/scratch2/weka/mcdermott/msaddler/swc/english/...,452.154921,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,written
374,tonyle,3.0,3.0,0.0,swc,0.31,1208.98,1208.67,/scratch2/weka/mcdermott/msaddler/swc/english/...,2359.540680,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,wrote


In [5]:
df_brir_room

,index,azim,elev,brir,sr
0,144,0,0,"[[-5.3024496e-06, -5.2993905e-06], [6.7206984e...",50000
1,146,10,0,"[[-6.6247826e-06, -1.4650306e-06], [1.2324836e...",50000
2,148,20,0,"[[8.585003e-06, -1.9858771e-07], [-1.5101928e-...",50000
3,150,30,0,"[[6.4137785e-06, 3.9158486e-08], [1.5993201e-0...",50000
4,152,40,0,"[[-1.0611512e-05, -1.2110121e-07], [1.0505603e...",50000
...,...,...,...,...,...
90,494,310,40,"[[-4.1528313e-07, 1.09590155e-05], [5.154498e-...",50000
91,496,320,40,"[[-7.2134935e-07, -1.1430697e-05], [7.4756053e...",50000
92,498,330,40,"[[-1.04854e-06, -1.1661021e-05], [8.016529e-07...",50000
93,500,340,40,"[[-1.521869e-06, 8.1888e-06], [4.2703056e-07, ...",50000


In [6]:
all_data = [read(fn) for fn in word_data['src_fn']]

In [7]:
all_words_resampled = [soxr.resample(data[1].astype(np.float32), data[0], 50000) for data in all_data]

In [8]:
all_words_resampled = [torch.Tensor(data) for data in all_words_resampled]
all_words = torch.stack(all_words_resampled)

In [9]:
irs = df_brir_room['brir'].values
irs = [torch.flip(torch.Tensor(ir), dims=[0]) for ir in irs]
irs = torch.stack(irs)

In [15]:
spatial_test = torch.nn.functional.conv1d(all_words_resampled[5].view(5, 1, -1).cuda(), irs[0].view(2, 1, -1).cuda()).cuda()

In [16]:
spatial_test.view(5, 2, -1).shape

torch.Size([5, 2, 5001])

In [18]:
sounds = mass_spatialize(all_words[:5], irs[0])

In [19]:
sounds.shape

torch.Size([5, 2, 150000])

In [10]:
def mass_spatialize(words, ir):
    """Uses pytorch to convolve all sounds in words with 2 channel IR given in ir"""
    n_words = words.shape[0]
    words_padded = [torch.nn.functional.pad(word, (ir.shape[0] - 1, 0)) for word in words]
    ir = ir.T.unsqueeze(1)
    words_padded = torch.stack(words_padded)
    spatialized = torch.nn.functional.conv1d(words_padded.view(n_words, 1, -1).cuda(), ir.cuda()).cuda()
    return spatialized

In [11]:
!nvidia-smi

Thu Jun 29 15:50:48 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 515.86.01    Driver Version: 515.86.01    CUDA Version: 11.7     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA A100 80G...  On   | 00000000:C3:00.0 Off |                    0 |
| N/A   39C    P0    76W / 300W |   3793MiB / 81920MiB |      0%      Default |
|                               |                      |             Disabled |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [12]:
n_words = all_words.shape[0]
n_irs = irs.shape[0]
with h5py.File('first_376.h5', 'w') as h5:
    for i in tqdm.tqdm(range(n_irs)):
        azim = df_brir_room['azim'][i]
        elev = df_brir_room['elev'][i]
        brir = irs[i]
        loc_group = h5.create_group(f'{azim}_{elev}')
        talker_id_kernel = np.array(word_data['client_id'], dtype='object')
        word_kernel = np.array(word_data['word'], dtype='object')
        signal_kernel = mass_spatialize(all_words, brir)
        loc_group.create_dataset('talker_id', data=talker_id_kernel)
        loc_group.create_dataset('word', data=word_kernel)
        loc_group.create_dataset('signal', data=signal_kernel.cpu().numpy())

100%|██████████| 95/95 [04:37<00:00,  2.92s/it]


In [13]:
h5 = h5py.File('first_376.h5', 'r')

In [14]:
h5.keys()

<KeysViewHDF5 ['0_0', '0_10', '0_20', '0_30', '0_40', '10_0', '10_10', '10_20', '10_30', '10_40', '20_0', '20_10', '20_20', '20_30', '20_40', '270_0', '270_10', '270_20', '270_30', '270_40', '280_0', '280_10', '280_20', '280_30', '280_40', '290_0', '290_10', '290_20', '290_30', '290_40', '300_0', '300_10', '300_20', '300_30', '300_40', '30_0', '30_10', '30_20', '30_30', '30_40', '310_0', '310_10', '310_20', '310_30', '310_40', '320_0', '320_10', '320_20', '320_30', '320_40', '330_0', '330_10', '330_20', '330_30', '330_40', '340_0', '340_10', '340_20', '340_30', '340_40', '350_0', '350_10', '350_20', '350_30', '350_40', '40_0', '40_10', '40_20', '40_30', '40_40', '50_0', '50_10', '50_20', '50_30', '50_40', '60_0', '60_10', '60_20', '60_30', '60_40', '70_0', '70_10', '70_20', '70_30', '70_40', '80_0', '80_10', '80_20', '80_30', '80_40', '90_0', '90_10', '90_20', '90_30', '90_40']>